In [2]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np

df1 = pd.read_csv('dataset/scenario-1-disrupt.csv', low_memory=False)
df2 = pd.read_csv('dataset/scenario-2-disrupt.csv', low_memory=False)
df3 = pd.read_csv('dataset/scenario-3-disrupt.csv', low_memory=False)
df4 = pd.read_csv('dataset/scenario-4-disrupt.csv', low_memory=False)
df5 = pd.read_csv('dataset/scenario-5-disrupt.csv', low_memory=False)
df6 = pd.read_csv('dataset/scenario-6-authtest.csv', low_memory=False)
df7 = pd.read_csv('dataset/scenario-7-authtest.csv', low_memory=False)
df8 = pd.read_csv('dataset/scenario-8-authtest.csv', low_memory=False)

df = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8])

counts = df['SubCategoryLabel'].value_counts()
summary = pd.DataFrame({
    'Count': counts,
    'Percent': (counts / counts.sum() * 100).round(2)
})
summary.loc['Total'] = [counts.sum(), 100.0]
summary['Count'] = summary['Count'].astype(int)
print(summary)


                   Count  Percent
SubCategoryLabel                 
dos-udp           131072    16.05
dos-icmp          131072    16.05
dos-pushack       130942    16.04
user              125128    15.33
dos-slowloris      85212    10.44
background         83244    10.20
dos-hulk           51327     6.29
admin              32664     4.00
recon-nmap         27713     3.39
bruteforce-smb     10001     1.22
bruteforce-ssh      4696     0.58
bruteforce-ftp      3344     0.41
recon-dns             20     0.00
Total             816435   100.00


In [4]:
# === Preprocessing ===

# 1. Define features to keep
numeric_features = [
    'DstJitter','DstTCPBase','SAppBytes','DIntPktMin','SrcJitAct','Offset',
    'DstBytes','DstLoad','SrcBytes','SrcLoss','TotPkts','SrcTCPBase','Loss',
    'Rate','SIntPktIdl','RunTime','DstLoss','sMeanPktSz','TotAppByte','Sdaddr',
    'SrcPkts','TotBytes','DstRate','Dport','dMaxPktSz','Max','SrcLoad','Sport',
    'Mean','TcpRtt','PCRatio','pLoss','Ssaddr','sMaxPktSz','DstWin','SIntPktMin',
    'Sum','sTos','SIntPkt','sHops','Dur','DIntPkt','SynAck','AckDat','SIntPktMax',
    'DIntPktMax','SrcRate','Load','Seq','dMinPktSz','sTtl','Min','SIntPktAct',
    'DIntPktAct','SrcJitter','dMeanPktSz','SrcWin','sMinPktSz','DAppBytes','DstPkts'
]

onehot_features = ['State', 'Flgs', 'Proto']
label_col = 'SubCategoryLabel'

# 2. Select only the columns we need
df_selected = df[numeric_features + onehot_features + [label_col]].copy()

# 3. Replace non-numeric placeholders (e.g. whitespace, empty) with NaN, then fill with 0
for col in numeric_features:
    df_selected[col] = pd.to_numeric(df_selected[col], errors='coerce').fillna(0)

# 4. One-hot encode State, Flgs, Proto
df_processed = pd.get_dummies(df_selected, columns=onehot_features, dtype=int)

# 5. Separate features and label
y = df_processed[label_col]
X = df_processed.drop(columns=[label_col])

print(f"Features shape: {X.shape}")
print(f"Label shape:    {y.shape}")
print(f"\nNumeric features:    {len(numeric_features)}")
print(f"One-hot columns:     {X.shape[1] - len(numeric_features)}")
print(f"Total feature cols:  {X.shape[1]}")
print(f"\nLabel distribution:")
print(y.value_counts())


Features shape: (816435, 84)
Label shape:    (816435,)

Numeric features:    60
One-hot columns:     24
Total feature cols:  84

Label distribution:
SubCategoryLabel
dos-udp           131072
dos-icmp          131072
dos-pushack       130942
user              125128
dos-slowloris      85212
background         83244
dos-hulk           51327
admin              32664
recon-nmap         27713
bruteforce-smb     10001
bruteforce-ssh      4696
bruteforce-ftp      3344
recon-dns             20
Name: count, dtype: int64


In [5]:
df_processed.to_csv('dataset/preprocessed_dataset.csv', index=False)
print(f"Saved {len(df_processed)} rows to dataset/preprocessed_dataset.csv")

Saved 816435 rows to dataset/preprocessed_dataset.csv


In [7]:
from sklearn.model_selection import train_test_split

label_col = 'SubCategoryLabel'
feature_col = [c for c in df_processed.columns if c not in label_col]
X = df_processed[feature_col]
y = df_processed[label_col]

(X_train, X_test, y_train, y_test) = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

all_subs = set(y.unique())
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Sub-categories in train: {len(set(y_train.unique()))}/13")
print(f"Sub-categories in test : {len(set(y_test.unique()))}/13")
print(f"\n{'Sub-category':<22} {'Train':>8} {'Test':>7} {'Test%':>7}")
print('-' * 48)
for sub in sorted(all_subs):
    tr = (y_train == sub).sum()
    te = (y_test  == sub).sum()
    print(f"{sub:<22} {tr:>8,} {te:>7,} {te/(tr+te)*100:>6.1f}%")

Train: 653,148  |  Test: 163,287
Sub-categories in train: 13/13
Sub-categories in test : 13/13

Sub-category              Train    Test   Test%
------------------------------------------------
admin                    26,131   6,533   20.0%
background               66,595  16,649   20.0%
bruteforce-ftp            2,675     669   20.0%
bruteforce-smb            8,001   2,000   20.0%
bruteforce-ssh            3,757     939   20.0%
dos-hulk                 41,061  10,266   20.0%
dos-icmp                104,858  26,214   20.0%
dos-pushack             104,754  26,188   20.0%
dos-slowloris            68,170  17,042   20.0%
dos-udp                 104,858  26,214   20.0%
recon-dns                    16       4   20.0%
recon-nmap               22,170   5,543   20.0%
user                    100,102  25,026   20.0%


In [ ]:
import os
OUT_DIR = 'dataset/splits'
os.makedirs(OUT_DIR, exist_ok=True)
def save(obj, name):
    path = f'{OUT_DIR}/{name}'
    obj.to_csv(path, index=False)
    print(f"  {name:<35} {len(obj):>8,} rows  {os.path.getsize(path)/1e6:>7.1f} MB")
print(f"Saving to {OUT_DIR}/ ...")
save(X_train,                               'X_train.csv')
save(X_test,                                'X_test.csv')
save(y_train,     'y_train.csv')
save(y_test,      'y_test.csv') 


Saving to dataset/splits/ ...
  X_train.csv                          653,148 rows    265.3 MB
  X_test.csv                           163,287 rows     66.3 MB
  y_train.csv                          653,148 rows      6.9 MB
  y_test.csv                           163,287 rows      1.7 MB
